In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

data = [
    (1, "A100", "2026-07-25 10:15:00", "2026-07-25", "2026-07-25 10:30:00", 500),
    (2, "A101", "2026-07-26 12:20:00", "2026-07-26", "2026-07-26 12:40:00", 700),
    (3, "A102", "2026-07-26 12:20:00", "2026-07-26", "2026-07-26 12:45:00", 700),
    (4, "A103", "2026-07-27 09:05:00", "2026-07-27", "2026-07-27 09:25:00", 300),
    (5, "A104", "2026-07-28 18:55:00", "2026-07-28", "2026-07-28 19:10:00", 900),
    (6, "A105", None,                  "2026-07-29", "2026-07-29 08:00:00", 450),
]

columns = [
    "id",
    "order_id",
    "event_ts_str",
    "event_date_str",
    "ingestion_ts_str",
    "amount"
]

df = spark.createDataFrame(data, columns)
df.show(truncate=False)
df.printSchema()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

df2 = (
    df
    .withColumn("event_ts", to_timestamp("event_ts_str", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("event_date", to_date("event_date_str", "yyyy-MM-dd"))
    .withColumn("ingestion_ts", to_timestamp("ingestion_ts_str", "yyyy-MM-dd HH:mm:ss"))
)

In [0]:
df2.select("event_ts_str", "event_ts", "event_date_str", "event_date").show(truncate=False)

In [0]:
df2.select(
    "order_id",
    year("event_ts").alias("year"),
    month("event_ts").alias("month"),
    dayofmonth("event_ts").alias("day"),
    hour("event_ts").alias("hour"),
    minute("event_ts").alias("minute")
).show()

In [0]:
df2.select(
    "order_id",
    date_format("event_ts", "dd-MM-yyyy HH:mm").alias("formatted_ts")
).show(truncate=False)

In [0]:
df2.withColumn("load_date", current_date()) \
   .withColumn("load_ts", current_timestamp()) \
   .select("order_id", "load_date", "load_ts") \
   .show(truncate=False)

In [0]:
df2.withColumn("days_diff", datediff("ingestion_ts", "event_ts")) \
   .select("order_id", "days_diff") \
   .show()

In [0]:
df2.select(
    "order_id",
    date_add("event_date", 7).alias("plus_7_days"),
    date_sub("event_date", 7).alias("minus_7_days")
).show(truncate=False)

In [0]:
df2.select(
    "order_id",
    trunc("event_date", "month").alias("first_day_month"),
    last_day("event_date").alias("last_day_month")
).show()

In [0]:
df2.select(
    "order_id",
    date_trunc("day", "event_ts").alias("trunc_day"),
    date_trunc("hour", "event_ts").alias("trunc_hour")
).show(truncate=False)

In [0]:
df2.filter(col("event_date") >= date_sub(current_date(), 7)) \
   .select("order_id", "event_date") \
   .show()

In [0]:
w = Window.partitionBy("order_id").orderBy(col("ingestion_ts").desc())

df_dedup = df2.withColumn("rn", row_number().over(w)) \
              .filter(col("rn") == 1) \
              .drop("rn")

df_dedup.select("order_id", "event_ts", "ingestion_ts").show(truncate=False)

In [0]:
df2.select(
    "order_id",
    when(col("ingestion_ts") > col("event_ts"), "delayed").otherwise("on_time").alias("status")
).show()

In [0]:
df2.withColumn("ist_ts", from_utc_timestamp("event_ts", "Asia/Kolkata")) \
   .select("order_id", "event_ts", "ist_ts") \
   .show(truncate=False)

In [0]:
df2.groupBy(to_date("event_ts").alias("sale_date")) \
   .agg(
       sum("amount").alias("total_sales"),
       count("*").alias("order_count")
   ).show()

In [0]:
stream_df = df2.withWatermark("event_ts", "10 minutes")

In [0]:

df2.select(
    "order_id",
    date_trunc("month", "event_ts").alias("month_ts"),
    date_trunc("day", "event_ts").alias("day_ts")
).show(truncate=False)